# FA-KGD Multicoil-ACS + T-Convergence Sweep — Colab runner

Two experiments:
1. **R-sweep** at fixed T=20: R∈{4,8,12,16} × {oracle, multicoil_acs}
2. **T-convergence**: T∈{10,20,50,100} × R∈{4,8} with ΠGDM and FA-KGD only

Noise model: **freq-dependent** (σ²(r) = σ_base·(1 + β·(r/r_max)²), β=5).
White (isotropic) noise collapses FA-KGD's frequency-adaptive Kalman gain
to a scalar identical to ΠGDM's, so the whole method is a no-op there.

**Prereqs in `MyDrive/fastmri_artifacts/`:**
- `network-snapshot.pkl` (250 MB)
- `brain_12vols.tar` (~3.6 GB)

**Runtime → T4 GPU.** Wall time ~3–4 h total.

In [ ]:
# Cell 1 — verify GPU
!nvidia-smi | head -20

In [ ]:
# Cell 2 — clone main repo (or git pull if already there); ensure ADPS source tree
# (ADPS provides dnnlib + torch_utils, which the EDM checkpoint pickle references.)
import os
%cd /content
if os.path.isdir('/content/fastmri/.git'):
    %cd /content/fastmri
    !git fetch --quiet && git reset --hard origin/main
else:
    !rm -rf fastmri
    !git clone https://github.com/carlo-scr/fastmri.git
    %cd /content/fastmri
if not os.path.isdir('external/adps/dnnlib'):
    !rm -rf external/adps
    !git clone --depth 1 https://github.com/utcsilab/ambient-diffusion-mri.git external/adps
!ls external/adps/dnnlib external/adps/torch_utils | head -5
!git --no-pager log -1 --oneline

In [ ]:
# Cell 3 — install deps + preflight checks
!pip install -q h5py scikit-image s3fs wandb pyyaml
# numpy guard (project notes flag 2.2.5 as broken)
!pip install -q --upgrade "numpy>=1.26,<2.3"

# Preflight: ensure Colab has the updated reconstruct.py with gated M-step CLI.
# If this fails, commit+push local changes and re-run Cell 2.
import subprocess
help_text = subprocess.run(
    ['python', 'scripts/reconstruct.py', '--help'],
    check=True, capture_output=True, text=True,
).stdout
assert '--m_step_start_frac' in help_text, (
    'Colab cloned an older fastmri commit that does not include --m_step_start_frac. '
    'Commit + push scripts/reconstruct.py and src/samplers/fakgd.py locally, then re-run Cell 2.'
)
print('OK: gated M-step CLI present')

In [ ]:
# Cell 4 — mount Drive and stage artifacts
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
ART = '/content/drive/MyDrive/fastmri_artifacts'
assert os.path.exists(f'{ART}/network-snapshot.pkl'), 'checkpoint missing in Drive'
assert os.path.exists(f'{ART}/brain_12vols.tar'),    'tarball missing in Drive'

os.makedirs('checkpoints/edm/supervised_R=1', exist_ok=True)
shutil.copy(f'{ART}/network-snapshot.pkl', 'checkpoints/edm/supervised_R=1/network-snapshot.pkl')

# Tarball preserves data/multicoil_val/... paths, so extract from repo root
!tar xf "$ART/brain_12vols.tar" -C /content/fastmri

# Strip macOS AppleDouble junk (._* / .DS_Store) that BSD tar on macOS injects;
# h5py would otherwise try to open them and fail with 'file signature not found'.
!find data/multicoil_val -name '._*' -delete
!find data/multicoil_val -name '.DS_Store' -delete

# Sanity-check every file actually opens as HDF5
import h5py, glob
files = sorted(glob.glob('data/multicoil_val/*.h5'))
bad = []
for f in files:
    try:
        with h5py.File(f, 'r'): pass
    except Exception as e:
        bad.append((f, str(e)))
print(f'{len(files)} h5 files; {len(bad)} unreadable')
for f, e in bad: print('  BAD:', f, '→', e)
!ls -la data/multicoil_val/ | head -15
!ls -la checkpoints/edm/supervised_R=1/

In [ ]:
# Cell 5 — smoke test: R=8, T=20, multicoil_acs init, FA-KGD with gated full M-step.
#
# m_step_mode=full + m_step_start_frac=0.5: the EMA only kicks in during the
# low-σ_t half of sampling, where the residual really *is* dominated by
# measurement noise (not denoiser error). This avoids the σ_i² blow-up that
# `clamp` was patching, while ALLOWING σ_i² to grow above the flat
# multicoil-ACS init at high frequencies — which is the only way FA-KGD's
# frequency-adaptive Kalman gain can do anything different from ΠGDM's
# scalar gain when the init is flat.
# γ>0 enables the bias-correction subtraction.
# Use freq_dep noise — white noise makes K isotropic and FA-KGD ≡ ΠGDM.
!python scripts/reconstruct.py \
  --mode edm \
  --checkpoint_dir checkpoints/edm/supervised_R=1 \
  --data_path data/multicoil_val \
  --num_slices 2 --num_steps 20 \
  --acceleration 8 --center_fraction 0.04 --schedule edm \
  --noise_init multicoil_acs --noise_model freq_dep --beta_noise 5.0 \
  --m_step_mode full --m_step_start_frac 0.5 \
  --gamma 0.05 --beta_fpdc 1.0 --alpha_ema 0.95 \
  --target_resolution 320 320 \
  --device cuda \
  --methods pigdm fakgd \
  --output_dir outputs/_smoke_mc

In [ ]:
# Cell 6 — R-sweep at T=20: oracle vs multicoil_acs
# Per-R center_fraction so high-R isn't capped by ACS lines
# (W=320, cf=0.08 -> 26 ACS lines, which already exceeds 320/16=20 lines).
#
# NOISE MODEL: freq_dep (sigma^2(r) = sigma_base*(1 + beta*(r/r_max)^2), beta=5 -> 6x variance
# contrast center->edge). White noise collapses FA-KGD's Kalman gain to a
# scalar identical to PiGDM's -> delta PSNR = 0 by construction.
#
# M-STEP: full + start_frac=0.5. The clamp+multicoil_acs combo is self-defeating:
# multicoil_acs init is effectively flat, and clamp forbids growth above init.
#
# IMPORTANT: use a unique output root so stale runs are not silently reused.
import subprocess, time, os
DATA='data/multicoil_val'; CKPT='checkpoints/edm/supervised_R=1'
T=20
NUM_SLICES = 12 * 16  # 12 brain volumes x 16 slices = 192; --whole_volume clamps
CF={4:0.08, 8:0.04, 12:0.04, 16:0.04}
RUN_TAG='fd_b5_full_s05_g005'
OUTROOT=f'outputs/Rsweep_T{T}_{RUN_TAG}'; os.makedirs(OUTROOT, exist_ok=True)
configs=[(R, init) for R in (4,8,12,16) for init in ('oracle','multicoil_acs')]
for R, init in configs:
    out=f'{OUTROOT}/R{R}_{init}'
    if os.path.exists(f'{out}/results.json'):
        print(f'  skip (exists): {out}'); continue
    print(f'\n===== R={R}  cf={CF[R]}  init={init}  -> {out} =====')
    t0=time.time()
    cmd=['python','-u','scripts/reconstruct.py',
        '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
        '--num_slices',str(NUM_SLICES),'--whole_volume',
        '--acceleration',str(R),'--center_fraction',str(CF[R]),
        '--num_steps',str(T),'--schedule','edm',
        '--noise_init',init,'--noise_model','freq_dep','--beta_noise','5.0',
        '--m_step_mode','full','--m_step_start_frac','0.5','--gamma','0.05',
        '--beta_fpdc','1.0','--alpha_ema','0.95',
        '--target_resolution','320','320','--device','cuda',
        '--methods','pigdm','fakgd',
        '--output_dir',out]
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        raise RuntimeError(f'reconstruct.py failed (rc={r.returncode}) for R={R} init={init}')
    print(f'  -> done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 7 — R-sweep summary table
import json, numpy as np
from pathlib import Path
T=20
RUN_TAG='fd_b5_full_s05_g005'
OUTROOT=Path(f'outputs/Rsweep_T{T}_{RUN_TAG}')
print(f"Using results root: {OUTROOT}")
print(f"{'R':>3} {'init':<14} {'PiGDM PSNR':>12} {'FA-KGD PSNR':>14} {'dPSNR':>8}  "
      f"{'PiGDM SSIM':>12} {'FA-KGD SSIM':>14} {'dSSIM':>9}")
print('-'*100)
for R in (4,8,12,16):
    for init in ('oracle','multicoil_acs'):
        p=OUTROOT/f'R{R}_{init}'/'results.json'
        if not p.exists():
            print(f'R={R} {init}: MISSING'); continue
        d=json.load(open(p)); pv=d['per_volume']
        pi_p=np.array([v['pigdm_psnr'] for v in pv.values()])
        fa_p=np.array([v['fakgd_psnr'] for v in pv.values()])
        pi_s=np.array([v['pigdm_ssim'] for v in pv.values()])
        fa_s=np.array([v['fakgd_ssim'] for v in pv.values()])
        print(f'{R:>3} {init:<14} {pi_p.mean():>7.2f}+/-{pi_p.std():.2f} '
              f'{fa_p.mean():>9.2f}+/-{fa_p.std():.2f} {(fa_p-pi_p).mean():>+8.2f}  '
              f'{pi_s.mean():>7.4f}+/-{pi_s.std():.4f} {fa_s.mean():>9.4f}+/-{fa_s.std():.4f} '
              f'{(fa_s-pi_s).mean():>+9.4f}')

In [ ]:
# Cell 8 — T-convergence sweep: T in {10,20,50,100} x R in {4,8}, multicoil_acs init
# noise_model=freq_dep + m_step_mode=full + start_frac=0.5: see Cell 6.
import subprocess, time, os
DATA='data/multicoil_val'; CKPT='checkpoints/edm/supervised_R=1'
NUM_SLICES = 12 * 16  # 12 brain volumes x 16 slices = 192
CF={4:0.08, 8:0.04}
RUN_TAG='fd_b5_full_s05_g005'
OUTROOT=f'outputs/Tsweep_{RUN_TAG}'; os.makedirs(OUTROOT, exist_ok=True)
for R in (4,8):
    for T in (10,20,50,100):
        out=f'{OUTROOT}/R{R}_T{T}'
        if os.path.exists(f'{out}/results.json'):
            print(f'  skip (exists): {out}'); continue
        print(f'\n===== R={R}  cf={CF[R]}  T={T}  -> {out} =====')
        t0=time.time()
        cmd=['python','-u','scripts/reconstruct.py',
            '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
            '--num_slices',str(NUM_SLICES),'--whole_volume',
            '--acceleration',str(R),'--center_fraction',str(CF[R]),
            '--num_steps',str(T),'--schedule','edm',
            '--noise_init','multicoil_acs','--noise_model','freq_dep','--beta_noise','5.0',
            '--m_step_mode','full','--m_step_start_frac','0.5','--gamma','0.05',
            '--beta_fpdc','1.0','--alpha_ema','0.95',
            '--target_resolution','320','320','--device','cuda',
            '--methods','pigdm','fakgd',
            '--output_dir',out]
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-6000:])
        if r.returncode != 0:
            raise RuntimeError(f'reconstruct.py failed (rc={r.returncode}) for R={R} T={T}')
        print(f'  -> done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 9 — T-convergence plot
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
RUN_TAG='fd_b5_full_s05_g005'
OUTROOT=Path(f'outputs/Tsweep_{RUN_TAG}')
Ts=[10,20,50,100]
fig, ax = plt.subplots(1, 1, figsize=(6,4))
for R, color in zip((4,8), ('C0','C3')):
    pi_means, fa_means = [], []
    for T in Ts:
        p=OUTROOT/f'R{R}_T{T}'/'results.json'
        if not p.exists(): pi_means.append(np.nan); fa_means.append(np.nan); continue
        d=json.load(open(p)); pv=d['per_volume']
        pi_means.append(np.mean([v['pigdm_psnr'] for v in pv.values()]))
        fa_means.append(np.mean([v['fakgd_psnr'] for v in pv.values()]))
    ax.plot(Ts, pi_means, '--o', color=color, label=f'PiGDM R={R}')
    ax.plot(Ts, fa_means, '-s',  color=color, label=f'FA-KGD R={R}')
ax.set_xscale('log'); ax.set_xlabel('Diffusion steps T'); ax.set_ylabel('PSNR (dB)')
ax.set_xticks(Ts); ax.set_xticklabels(Ts)
ax.grid(alpha=0.3); ax.legend(); ax.set_title('T-convergence: FA-KGD vs PiGDM')
fig_path = OUTROOT/'Tconvergence.png'
plt.tight_layout(); plt.savefig(fig_path, dpi=150); plt.show()
print('Saved:', fig_path)

In [ ]:
# Cell 10 — back up everything to Drive
import os, tarfile
ART = '/content/drive/MyDrive/fastmri_artifacts'  # self-contained
RUN_TAG='fd_b5_full_s05_g005'
out_tar = f'{ART}/sweep_results_{RUN_TAG}.tar'
with tarfile.open(out_tar, 'w') as tf:
    for d in (f'outputs/Rsweep_T20_{RUN_TAG}', f'outputs/Tsweep_{RUN_TAG}'):
        if os.path.exists(d):
            tf.add(d, arcname=os.path.basename(d))
            print(f'  + {d}')
        else:
            print(f'  - {d} (missing, skipping)')
print('Saved:', out_tar)
!ls -la "$ART" 2>/dev/null || ls -la /content/drive/MyDrive/fastmri_artifacts